In [ ]:
import numpy as np 
import polars as pl 
import polars.selectors as cs
import pandas as pd 
import utils

In [ ]:
# import data
train = pl.read_parquet("../data/cleaned/application_train.parquet")
test  = pl.read_parquet("../data/cleaned/application_test.parquet")
buro  = pl.read_parquet("../data/cleaned/bureau.parquet")
bbal  = pl.read_parquet("../data/cleaned/bureau_balance.parquet")
prev  = pl.read_parquet("../data/cleaned/previous_application.parquet")
card  = pl.read_parquet("../data/cleaned/credit_card_balance.parquet")
poca  = pl.read_parquet("../data/cleaned/POS_CASH_balance.parquet")
inst  = pl.read_parquet("../data/cleaned/installments_payments.parquet")

In [ ]:
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# garbage collection
import gc
gc.enable()

In [ ]:
# check dimensions
print("Application:", train.shape, test.shape)
print("Buro:", buro.shape)
print("Bbal:", bbal.shape)
print("Prev:", prev.shape)
print("Card:", card.shape)
print("Poca:", poca.shape)
print("Inst:", inst.shape)

In [ ]:
# extract target
y = train.select(["SK_ID_CURR", "TARGET"])
train = train.drop("TARGET")

In [ ]:
# concatenate application data
appl = pl.concat([train, test])
del train, test

In [ ]:
import polars as pl
import utils

# list of documents
doc_vars = ["FLAG_DOCUMENT_2",  "FLAG_DOCUMENT_3",  "FLAG_DOCUMENT_4",  "FLAG_DOCUMENT_5",  "FLAG_DOCUMENT_6",
            "FLAG_DOCUMENT_7",  "FLAG_DOCUMENT_8",  "FLAG_DOCUMENT_9",  "FLAG_DOCUMENT_10", "FLAG_DOCUMENT_11",
            "FLAG_DOCUMENT_12", "FLAG_DOCUMENT_13", "FLAG_DOCUMENT_14", "FLAG_DOCUMENT_15", "FLAG_DOCUMENT_16",
            "FLAG_DOCUMENT_17", "FLAG_DOCUMENT_18", "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21"]

# 1. Feature Engineering (computed in parallel)
appl = appl.with_columns(
    # income ratios
    (pl.col("AMT_CREDIT") / pl.col("AMT_INCOME_TOTAL")).alias("CREDIT_BY_INCOME"),
    (pl.col("AMT_ANNUITY") / pl.col("AMT_INCOME_TOTAL")).alias("ANNUITY_BY_INCOME"),
    (pl.col("AMT_GOODS_PRICE") / pl.col("AMT_INCOME_TOTAL")).alias("GOODS_PRICE_BY_INCOME"),
    (pl.col("AMT_INCOME_TOTAL") / pl.col("CNT_FAM_MEMBERS")).alias("INCOME_PER_PERSON"),
    
    # career ratio (replaces negatives with None)
    pl.when((pl.col("DAYS_EMPLOYED") / pl.col("DAYS_BIRTH")) < 0)
      .then(None)
      .otherwise(pl.col("DAYS_EMPLOYED") / pl.col("DAYS_BIRTH"))
      .alias("PERCENT_WORKED"),
      
    # number of adults and children ratio
    (pl.col("CNT_FAM_MEMBERS") - pl.col("CNT_CHILDREN")).alias("CNT_ADULTS"),
    (pl.col("CNT_CHILDREN") / pl.col("CNT_FAM_MEMBERS")).alias("CHILDREN_RATIO"),
    
    # overall payments
    (pl.col("AMT_CREDIT") / pl.col("AMT_ANNUITY")).alias("ANNUITY LENGTH"),
    
    # external sources
    pl.mean_horizontal("EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3").alias("EXT_SOURCE_MEAN"),
    # sum_horizontal on is_not_null() perfectly mimics the "3 - sum(is_null)" logic natively
    pl.sum_horizontal(
        pl.col("EXT_SOURCE_1").is_not_null(),
        pl.col("EXT_SOURCE_2").is_not_null(),
        pl.col("EXT_SOURCE_3").is_not_null()
    ).alias("NUM_EXT_SOURCES"),
    
    # number of documents
    pl.sum_horizontal(doc_vars).alias("NUM_DOCUMENTS"),
    
    # application date (Weekend / Working day)
    pl.when(pl.col("WEEKDAY_APPR_PROCESS_START").is_in(["SATURDAY", "SUNDAY"]))
      .then(pl.lit("Weekend"))
      .otherwise(pl.lit("Working day"))
      .alias("DAY_APPR_PROCESS_START"),
      
    # age ratios
    (pl.col("OWN_CAR_AGE") / pl.col("DAYS_BIRTH")).alias("OWN_CAR_AGE_RATIO"),
    (pl.col("DAYS_ID_PUBLISH") / pl.col("DAYS_BIRTH")).alias("DAYS_ID_PUBLISHED_RATIO"),
    (pl.col("DAYS_REGISTRATION") / pl.col("DAYS_BIRTH")).alias("DAYS_REGISTRATION_RATIO"),
    (pl.col("DAYS_LAST_PHONE_CHANGE") / pl.col("DAYS_BIRTH")).alias("DAYS_LAST_PHONE_CHANGE_RATIO")
)

# 2. Apply Custom Functions (These now work correctly with the Polars DataFrame)
log_vars = ["AMT_CREDIT", "AMT_INCOME_TOTAL", "AMT_GOODS_PRICE", "AMT_ANNUITY"]
appl = utils.create_logarithms(appl, log_vars, replace=True)

day_vars = ["DAYS_BIRTH", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_EMPLOYED", "DAYS_LAST_PHONE_CHANGE"]
appl = utils.convert_days(appl, day_vars, t=30, rounding=True, replace=True)

# 3. Drop unused features
drops = ['APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 
         'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI',
         'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI','YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
         'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'COMMONAREA_MODE','ELEVATORS_MODE', 'ENTRANCES_MODE', 
         'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 
         'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'TOTALAREA_MODE',  'YEARS_BEGINEXPLUATATION_MODE']

appl = appl.drop(drops)


In [ ]:
rename_mapping = {col: f"app_{col}" for col in appl.columns if col != "SK_ID_CURR"}
appl.rename(rename_mapping)

In [ ]:
# check data
appl.head()

In [ ]:
# count missings
nas = utils.count_missings(appl)
nas.head()

In [ ]:
# check bbal data
bbal.head()

In [ ]:
loan_score = (
    bbal
    .with_columns(
            pl.col("STATUS").replace({"X": None, "1": 1.0, "2": 2.0, "3": 3.0, "4": 4.0, "5": 5.0}, default=0.0
    ).alias("NUM_STATUS")
    )
    .with_columns(
        (pl.col("NUM_STATUS")/(pl.col("MONTHS_BALANCE").abs()+1)).alias("LOAN_SCORE")
    )
    .group_by("SK_ID_BUREAU")
    .agg(pl.col("LOAN_SCORE").sum())
)

bbal = bbal.to_dummies("STATUS")

In [ ]:
# count missings
nas = utils.count_missings(bbal)
nas.head()

In [ ]:
agg_bbal = (
    bbal
    .group_by("SK_ID_BUREAU")
    .agg(
        pl.col("MONTHS_BALANCE").count().alias("MONTH_COUNT"),
        cs.starts_with("STATUS_").mean()
    )
    .join(loan_score,on= "SK_ID_BUREAU",how = "left")
)

In [ ]:
# count missings
nas = utils.count_missings(agg_bbal)
nas.head()

In [ ]:
# check data
agg_bbal.head()

In [ ]:
# clear memory
del bbal

In [ ]:
# check buro data
buro.head()

In [ ]:
buro = buro.join(agg_bbal, how = "left", on = "SK_ID_BUREAU")

In [ ]:
# 1. Total bureau loans per applicant (Using the window function shortcut!)
buro = buro.with_columns(
    pl.len().over("SK_ID_CURR").alias("CNT_BURO_LOANS")
)

# 2. Ratios
buro = buro.with_columns(
    (pl.col("AMT_CREDIT_SUM_OVERDUE") / pl.col("AMT_ANNUITY")).alias("AMT_SUM_OVERDUE_RATIO_1"),
    (pl.col("AMT_CREDIT_SUM_OVERDUE") / pl.col("AMT_CREDIT_SUM")).alias("AMT_SUM_OVERDUE_RATIO_2"),
    (pl.col("AMT_CREDIT_MAX_OVERDUE") / pl.col("AMT_ANNUITY")).alias("AMT_MAX_OVERDUE_RATIO_1"),
    (pl.col("AMT_CREDIT_MAX_OVERDUE") / pl.col("AMT_CREDIT_SUM")).alias("AMT_MAX_OVERDUE_RATIO_2"),
    (pl.col("AMT_CREDIT_SUM_DEBT") / pl.col("AMT_CREDIT_SUM")).alias("AMT_SUM_DEBT_RATIO_1"),
    (pl.col("AMT_CREDIT_SUM_DEBT") / pl.col("AMT_CREDIT_SUM_LIMIT")).alias("AMT_SUM_DEBT_RATIO_2"),
)

# 3. Logarithms
log_vars = ["AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY"]
buro = utils.create_logarithms(buro, log_vars, replace=True)

# 4. Convert Days
day_vars = ["DAYS_CREDIT", "CREDIT_DAY_OVERDUE", "DAYS_CREDIT_ENDDATE", "DAYS_ENDDATE_FACT", "DAYS_CREDIT_UPDATE"]
buro = utils.convert_days(buro, day_vars, t=1, rounding=False, replace=True)

# 5. Remaining engineered features
buro = buro.with_columns(
    # Recency-weighted loan score
    (pl.col("LOAN_SCORE") / (pl.col("DAYS_CREDIT") / 12)).alias("WEIGHTED_LOAN_SCORE"),
    
    # Fixed typo: DAYS_ENDDATE_UPDATE -> DAYS_CREDIT_UPDATE
    (pl.col("DAYS_ENDDATE_FACT") - pl.col("DAYS_CREDIT_ENDDATE")).alias("DAYS_END_DIFF_1"),
    (pl.col("DAYS_CREDIT_UPDATE") - pl.col("DAYS_CREDIT_ENDDATE")).alias("DAYS_END_DIFF_2"),
    (pl.col("DAYS_CREDIT_ENDDATE") - pl.col("DAYS_CREDIT")).alias("DAYS_DURATION_1"),
    (pl.col("DAYS_ENDDATE_FACT") - pl.col("DAYS_CREDIT")).alias("DAYS_DURATION_2"),
    
    # Active/Closed/Bad counts
    # By summing a boolean condition, it naturally counts matching rows (and gracefully defaults to 0 if none match!)
    (pl.col("CREDIT_ACTIVE") == "Active").sum().over("SK_ID_CURR").alias("CNT_BURO_ACTIVE"),
    (pl.col("CREDIT_ACTIVE") == "Closed").sum().over("SK_ID_CURR").alias("CNT_BURO_CLOSED"),
    (pl.col("CREDIT_ACTIVE") == "Bad debt").sum().over("SK_ID_CURR").alias("CNT_BURO_BAD")
)
